In [1]:
import os
import pandas as pd
from datetime import datetime 
import duckdb
import unicodedata
import sys
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/20/26 17:14:02] INFO     Using                                                                  ]8;id=975530;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=105079;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\framewo                
                             rk\project\rich_logging.yml' as logging configuration.                                

[04/20/26 17:14:03] WARNING  c:\Users\User\miniconda3\envs\central\Lib\site-packages\requests\__ini ]8;id=176811;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=758459;file://c:\Users\User\miniconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             t__.py:113: RequestsDependencyWarning: urllib3 (2.6.1) or chardet                     
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[04/20/26 17:14:05] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=581389;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=343089;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


In [2]:
# Add the 'src' directory to the path
sys.path.append(os.path.abspath("src"))
import central_perm_flow.pipelines.data_processing.nodes as nodes_dproc
import central_perm_flow.pipelines.calac_activos_baj_grad.nodes as nodes_abg

# Importación de Fuentes

In [3]:
central_calaca_ext  = catalog.load('central_calendario_extendido')
central_calaca_uptoday  = catalog.load('central_calendario_extendido_uptoday')
central_estaca_sd = catalog.load('central_estaca_sd')

                    INFO     Loading data from central_calendario_extendido                    ]8;id=490487;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=723201;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_calendario_extendido_uptoday            ]8;id=998121;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=834761;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_estaca_sd (ParquetDataset)...           ]8;id=183761;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=174251;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

## Importación de parámetros

In [4]:
parameters = catalog.load("parameters")

                    INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=401690;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=953668;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [5]:
# bajas_calac
bajas_calac = parameters['bajas_calac']
graduados_calac = parameters['graduados_calac']

## Bajas

In [6]:
bajas_calendario_academico = nodes_abg.momento_baja(
    # 1. El DataFrame (Objeto en memoria)
    central_estaca_sd=central_estaca_sd, 
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 2. Valores extraídos de diccionarios de configuración
    central_col_fechadef=bajas_calac['central_col_fechadef'],
    central_col_fechatemp=parameters['bajas_calac']['central_col_fechatemp'],
    
    # 3. Nombre del dataset según el catálogo
    central_calaca= central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 4. Parámetros de transformación y cruce
    left_on=parameters['bajas_calac']['merge_left_on'],
    right_on=parameters['bajas_calac']['merge_right_on'],
    group_key=parameters['bajas_calac']['central_calaca_col_cohorte_inicial'],
    sort_cols=parameters['bajas_calac']['central_calaca_col_sort']
)

In [7]:
parameters['bajas_calac']['central_calaca_col_cohorte_inicial']

'fecha_ingreso'

In [8]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_desercion = (
    bajas_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico', 'fecha_ingreso', 'semana_acumulada', 'di']]
    .groupby(['cohorte_inicial','nivel_academico',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'di': 'sum'})
    .reset_index()
    .sort_values(by='di', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_desercion.style.bar(
    subset=['di'], 
    color='#f8766d', # Un rojo suave tipo "soft red"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,cohorte_inicial,nivel_academico,fecha_ingreso,semana_acumulada,di
10,2025 1b,posgrado,2025-05-12 00:00:00,3.000000,11
72,2026 1a,pregrado,2026-01-19 00:00:00,3.000000,11
78,2026 1b,posgrado,2026-03-16 00:00:00,1.000000,10
6,2025 1a,posgrado,2025-03-17 00:00:00,27.000000,10
32,2025 1c,posgrado,2025-07-07 00:00:00,23.000000,7
0,2025 1a,posgrado,2025-03-17 00:00:00,3.000000,5
69,2026 1a,posgrado,2026-01-19 00:00:00,3.000000,5
70,2026 1a,posgrado,2026-01-19 00:00:00,4.000000,5
62,2025 1e,pregrado,2025-10-27 00:00:00,8.000000,4
4,2025 1a,posgrado,2025-03-17 00:00:00,19.000000,4


In [9]:
bajas_calendario_academico = catalog.load('central_bajas_calendario_academico')

[04/20/26 17:14:06] INFO     Loading data from central_bajas_calendario_academico              ]8;id=15363;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=579366;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

## Graduados

In [10]:
graduados_calendario_academico = nodes_abg.momento_grado(
    # 1. Referencias a datasets/DataFrames
    central_estaca=central_estaca_sd,
    central_calaca=central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
   
    col_gi=parameters['graduados_calac']['graduation_col_gi'],
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    fallback_weeks=parameters['graduados_calac']['graduation_fallback_weeks'],
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left=parameters['graduados_calac']['graduation_join_keys_left'],
    join_right=parameters['graduados_calac']['graduation_join_keys_right']
)

In [11]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_graduacion = (
    graduados_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico','nivel', 'fecha_ingreso', 'semana_acumulada', 'gi']]
    .groupby(['cohorte_inicial','nivel_academico','nivel',  'fecha_ingreso', 'semana_acumulada'])
    .agg({'gi': 'sum'})
    .reset_index()
    .sort_values(by='gi', ascending=False)
    .head(100)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_graduacion.style.bar(
    subset=['gi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,cohorte_inicial,nivel_academico,nivel,fecha_ingreso,semana_acumulada,gi
3,2025 1c,posgrado,especializacion,2025-07-07 00:00:00,32,58
2,2025 1b,posgrado,especializacion,2025-05-12 00:00:00,32,47
0,2025 1a,posgrado,especializacion,2025-03-17 00:00:00,32,24
1,2025 1a,posgrado,maestria,2025-03-17 00:00:00,48,16


## Activos

In [12]:
activos_calendario_academico = nodes_abg.momento_activos(
   central_estaca=central_estaca_sd,
    central_calaca=central_calaca_uptoday.drop(columns= 'cohorte'),
    
    # 2. Configuración de lógica de negocio (Duraciones y Grado Inmediato)
    dict_duracion=parameters['graduados_calac']['dict_niveles_duracion'],
    col_di='di',
    col_gi='gi',
    
    # 3. Parámetro de seguridad (Semanas por defecto si falla el cálculo)
    fallback_weeks=None,
    
    # 4. Llaves para el cruce de tablas (Joins)
    join_left='fecha_activo',
    join_right='fecha_inicio',
    group_key = 'fecha_ingreso'
) 

In [13]:
# 1. Obtenemos el Top 10 (mismo proceso que antes)
top_diez_eng = (
    activos_calendario_academico
    .loc[:, ['cohorte_inicial','nivel_academico','nivel','programa', 'fecha_ingreso', 'semana_acumulada','max_semana_teorica', 'engi']]
    .groupby(['cohorte_inicial','nivel_academico','nivel','programa',  'fecha_ingreso', 'semana_acumulada','max_semana_teorica'])
    .agg({'engi': 'sum'})
    .reset_index()
    .sort_values(by='engi', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_eng.style.bar(
    subset=['engi'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,cohorte_inicial,nivel_academico,nivel,programa,fecha_ingreso,semana_acumulada,max_semana_teorica,engi
3,2025 1c,posgrado,especializacion,auditoria y control,2025-07-07 00:00:00,32,32,4
0,2025 1a,posgrado,maestria,analitica de datos,2025-03-17 00:00:00,53,64,0
1,2025 1b,posgrado,maestria,analitica de datos,2025-05-12 00:00:00,46,64,0
2,2025 1b,posgrado,maestria,aseguramiento y auditoria de informacion,2025-05-12 00:00:00,46,48,0
4,2025 1c,posgrado,maestria,analitica de datos,2025-07-07 00:00:00,38,64,0
5,2025 1c,posgrado,maestria,aseguramiento y auditoria de informacion,2025-07-07 00:00:00,38,48,0
6,2025 1c,pregrado,pregrado,economia y finanzas,2025-07-07 00:00:00,38,128,0
7,2025 1c,pregrado,pregrado,ingenieria de sistemas y computacion,2025-07-07 00:00:00,38,128,0
8,2025 1d,posgrado,especializacion,auditoria y control,2025-09-01 00:00:00,30,32,0
9,2025 1d,posgrado,maestria,analitica de datos,2025-09-01 00:00:00,30,64,0


In [20]:
top_diez_activos = (
    activos_calendario_academico
    .groupby(['cohorte_inicial','fecha_ingreso','nivel_academico','nivel','programa',  'semana_acumulada','max_semana_teorica'])
    .agg(activos=('ai', 'sum')) # <-- Definimos: nombre = (columna, operacion)
    .reset_index()
    .sort_values(by='activos', ascending=False)
    .head(10)
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_activos.style.bar(
    subset=['activos'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook
top_diez_estilado

,cohorte_inicial,fecha_ingreso,nivel_academico,nivel,programa,semana_acumulada,max_semana_teorica,activos
17,2025 1e,2025-10-27 00:00:00,pregrado,pregrado,derecho,22,144,92
28,2026 1a,2026-01-19 00:00:00,pregrado,pregrado,derecho,14,144,85
39,2026 1b,2026-03-16 00:00:00,pregrado,pregrado,derecho,6,144,74
19,2026 1a,2026-01-19 00:00:00,posgrado,especializacion,alta gerencia,14,32,59
4,2025 1c,2025-07-07 00:00:00,posgrado,maestria,analitica de datos,38,64,52
16,2025 1e,2025-10-27 00:00:00,pregrado,pregrado,contaduria publica,22,128,51
8,2025 1d,2025-09-01 00:00:00,posgrado,especializacion,auditoria y control,30,32,49
1,2025 1b,2025-05-12 00:00:00,posgrado,maestria,analitica de datos,46,64,48
32,2026 1b,2026-03-16 00:00:00,posgrado,especializacion,ciencias tributarias,6,32,47
21,2026 1a,2026-01-19 00:00:00,posgrado,especializacion,ciencias tributarias,14,32,47


In [16]:
def consolidar_universo_academico(
    df_bajas: pd.DataFrame,
    df_graduados: pd.DataFrame,
    df_activos: pd.DataFrame
) -> pd.DataFrame:
    """
    Concatena los tres estados académicos y normaliza indicadores.
    """
    # Unimos los tres dataframes
    universo = pd.concat([df_bajas, df_graduados, df_activos], ignore_index=True)
    
    # Normalizamos la columna engi: si es nula (viene de bajas/grados), es 0
    if 'engi' in universo.columns:
        universo['engi'] = universo['engi'].fillna(0).astype(int)
        
    return universo

In [17]:
df_bajas = catalog.load('central_bajas_calendario_academico')
df_graduados = catalog.load('central_graduados_calendario_academico')
df_activos = catalog.load('central_activos_calendario')

                    INFO     Loading data from central_bajas_calendario_academico              ]8;id=934225;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=519019;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_graduados_calendario_academico          ]8;id=403140;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=387305;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\
                             (ParquetDataset)...                                                                   

                    INFO     Loading data from central_activos_calendario (ParquetDataset)...  ]8;id=508811;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=615658;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [18]:
df_consolidado = catalog.load('central_estados_calac')

                    INFO     Loading data from central_estados_calac (ParquetDataset)...       ]8;id=522351;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=767478;file://c:\Users\User\miniconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [19]:
df_consolidado['semana_acumulada']


0        3.0
1        3.0
2        3.0
3        3.0
4        3.0
        ... 
1750    22.0
1751    22.0
1752    22.0
1753    22.0
1754     6.0
Name: semana_acumulada, Length: 1755, dtype: float64

In [31]:
top_diez_activos = (
    df_consolidado
    .groupby(['cohorte_inicial','fecha_ingreso'])
    .agg(estudiantes=('identificacion', 'count')) # <-- Definimos: nombre = (columna, operacion)
    .reset_index()
    .sort_values(by='estudiantes', ascending=False)
  
)

# 2. Aplicamos las barras de datos de color rojo
top_diez_estilado = top_diez_activos.style.bar(
    subset=['estudiantes'], 
    color="#41747A", # Un verde suave tipo "soft green"
    align='left',    # Las barras nacen desde la izquierda
    vmin=0           # El mínimo es cero para que la escala sea real
)

# Visualizar en el Notebook


In [32]:
top_diez_activos.sort_values(by= 'fecha_ingreso').head(20)

,cohorte_inicial,fecha_ingreso,estudiantes
0,2025 1a,2025-03-17,110
1,2025 1b,2025-05-12,155
2,2025 1c,2025-07-07,206
3,2025 1d,2025-09-01,132
4,2025 1e,2025-10-27,296
5,2026 1a,2026-01-19,456
6,2026 1b,2026-03-16,397
